# Transform Races Data

1. Read bronze `races` table
1. Keep only the columns required for analytics (Drop `url` column)
1. Standardise column names using snake_case (`raceName` → `race_name`, `circuitId` → `circuit_id`)
1. Rename columns to make them more meaningful (`date` → `race_date`)
1. Remove duplicate records
1. Transform values of column `race_name` to Title Case
1. Write the transformed data to silver `races` table

In [0]:
%run "/Workspace/Users/deepanshu.patil69@gmail.com/Formula1/common/01_Environmnet_config"

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.races"
silver_table = f"{catalog_name}.{silver_schema}.races"

In [0]:
races_df = spark.read.table(bronze_table)
display(races_df)

In [0]:
races_drop_col_df = (
    races_df.drop("url")
)
display(races_drop_col_df)

In [0]:
races_renamed_col_df = (
    races_drop_col_df
        .withColumnsRenamed({
            "raceName":"race_name",
            "circuitId":"circuit_id",
            "date":"race_date"
        })
)
races_renamed_col_df.display()

In [0]:
display(
    races_renamed_col_df.select([
        F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
        for c in races_renamed_col_df.columns
    ])
)

In [0]:
display(
    races_renamed_col_df
        .groupBy(F.col('round'), F.col('season'))
        .agg(F.count('*').alias("count"))
        .filter(F.col("count") > 1)
)

In [0]:
races_drop_duplicates_df = (
    races_renamed_col_df
        .dropDuplicates(['round', 'season'])
)

races_drop_duplicates_df.display()

In [0]:
races_final_df = (
    races_drop_duplicates_df
        .withColumn('race_name', F.initcap(F.col('race_name')))
)
races_final_df.display()

In [0]:
(
    races_final_df
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(silver_table)
)

In [0]:
%sql

select * from formula1.silver.races;